[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/Zgraph/blob/main/zgraph/examples/binary.ipynb)

In [1]:
# Install Zgraph if running in Google Colab
try:
    import zgraph
    print("Zgraph is already installed.")
except ImportError:
    print("Installing Zgraph...")
    !pip install -q "git+https://github.com/themintlab/Zgraph.git#subdirectory=zgraph"
    print("Successfully installed Zgraph!")

Zgraph is already installed.


In [2]:
import torch
from zgraph import *
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
T, mu1, mu2 = SignalNodes(0,1,2)

In [4]:
R = 8.314
RT = FactorNode([[R]], [T])
mu1A = FactorNode([2, -1], [RT, mu1] )
mu2A = FactorNode([-1], [mu2])
mu1B = FactorNode([-1], [mu1])
mu2B = FactorNode([1, -1], [RT, mu2] )

In [5]:
phaseA = FactorNode(torch.eye(2), [mu1A, mu2A], beta=RT)
phaseB = FactorNode(torch.eye(2), [mu1B, mu2B], beta=RT)
system = FactorNode(torch.eye(2), [phaseA, phaseB], beta=0)
fcns = [phaseA, phaseB, system]

In [6]:
lt_fcns = legendre_transform(fcns, [1, 2])

In [7]:
# Compile graphs for execution
fcns_compiled = graph_to_function(fcns, compile=True)
lt_fcns_compiled = graph_to_function(lt_fcns, compile=True)

# Input signals

In [8]:
T_val = torch.tensor(298.15)
mu1 = torch.linspace(-10*R*300, 10*R*300, steps=500)
mu2 = -mu1 #torch.zeros_like(mu1)
T_flat = T_val.expand_as(mu1)
inputs = torch.stack([T_flat, mu1, mu2], dim=-1)

In [9]:
# Shift the coordinates to the equilibrium manifold
shifted_inputs = gauge_fix(fcns_compiled[2], inputs, [1, 2])

## Generate data

In [10]:
# Evaluate and strip PyTorch tracking in one clean pass
f_vals_list = to_numpy([f(shifted_inputs) for f in fcns_compiled])
lt_vals_list = to_numpy([f(shifted_inputs) for f in lt_fcns_compiled])

In [11]:
x_mu = to_numpy(inputs[..., 1])

# Create stacked subplots
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=False,
    vertical_spacing=0.12,
    subplot_titles=("Grand potential plot", "Free energy plot")
)

names = ["phase A", "phase B", "Equilibrium"]
colors = {
    "phase A": "#1f77b4",
    "phase B": "#ff7f0e",
    "Equilibrium": "#2ca02c"
}

# Zip everything together to loop over phases and the system!
for name, g_vals, (f_val, mu_val) in zip(names, f_vals_list, lt_vals_list):
    
    # --- Top Plot: Grand potential ---
    fig.add_trace(
        go.Scatter(
            x=x_mu, y=g_vals,
            name=name,
            legendgroup=name,
            showlegend=True,
            line=dict(width=3, color=colors[name])
        ),
        row=1, col=1
    )
    
    # --- Bottom Plot: Free energy ---
    x_frac = -mu_val[:, 1]
    y_free = -f_val
    
    fig.add_trace(
        go.Scatter(
            x=x_frac, y=y_free,
            name=name,
            legendgroup=name,
            showlegend=False,
            mode="lines+markers",
            marker=dict(size=4),
            line=dict(width=3, color=colors[name])
        ),
        row=2, col=1
    )

fig.update_xaxes(title_text="Chemical potential difference, Δμ", row=1, col=1)
fig.update_yaxes(title_text="Grand potential, Ω", row=1, col=1)

fig.update_xaxes(title_text="Mole fraction", row=2, col=1)
fig.update_yaxes(title_text="Free energy", row=2, col=1)

fig.update_layout(height=750, width=800, template="plotly_white")
fig.show()